# Ava quickstart

Corpus to a sampling model, end to end. Everything here is language-agnostic:
point step 1 at any Hugging Face dataset with a text column.

Runs on a single 16 GB GPU with the `hybrid-130m` preset. On CPU, drop
`block_size` and `max_steps` to something tiny just to see the loop move.


In [ ]:
# In Colab, uncomment to install:
# !pip install -q "ava-llm[all] @ git+https://github.com/Kuduxaaa/ava-llm"

import torch
from torch.utils.data import DataLoader

from ava import AvaConfig, AvaForCausalLM, GenerationConfig
from ava.data import PackedDataset, collate_packed, iter_text_file, pack_corpus
from ava.tokenizer import AvaTokenizer
from ava.training import TrainingConfig, train_model
from ava.utils import model_summary, set_seed

set_seed(1337)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"gpu:    {props.name} | {props.total_memory / 1e9:.1f} GB | bf16={torch.cuda.is_bf16_supported()}")

## 1. Corpus

One document per line, plain text. Any language, any script.

```bash
python scripts/download_corpus.py --dataset HuggingFaceFW/fineweb \
    --config sample-10BT --max-docs 200000
```

Set `CORPUS` to your own file if you already have one.

In [ ]:
from pathlib import Path

CORPUS = Path("data/corpus.txt")

if not CORPUS.exists():
    # A tiny stand-in so the notebook runs end to end without a download.
    CORPUS.parent.mkdir(parents=True, exist_ok=True)
    sample = [
        "A state space model carries a fixed size hidden state through time.",
        "Attention compares every position with every other position.",
        "A hybrid stack uses both, and pays for attention only where it helps.",
    ]
    CORPUS.write_text("\n".join(sample * 400), encoding="utf-8")
    print("wrote a placeholder corpus -- replace it with real data")

num_lines = sum(1 for _ in open(CORPUS, encoding="utf-8"))
print(f"{CORPUS}: {num_lines:,} documents, {CORPUS.stat().st_size / 1e6:.1f} MB")

## 2. Tokenizer

`character_coverage` is the one script-dependent knob: `1.0` for a small
alphabet, `0.9995` when there is a long tail of rare characters.

In [ ]:
TOKENIZER_DIR = Path("data/tokenizer")
VOCAB_SIZE = 8000  # raise to 32000 for a real corpus

if (TOKENIZER_DIR / "tokenizer_config.json").exists():
    tokenizer = AvaTokenizer.from_pretrained(TOKENIZER_DIR)
else:
    tokenizer = AvaTokenizer.train(
        CORPUS, TOKENIZER_DIR,
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=0.9995,
    )

sample = next(iter_text_file(CORPUS))
pieces = tokenizer.tokenize(sample)
print(f"vocab      {len(tokenizer):,}")
print(f"sample     {sample[:70]}")
print(f"pieces     {pieces[:12]}")
print(f"fertility  {len(pieces) / max(1, len(sample.split())):.2f} tokens/word")
assert tokenizer.decode(tokenizer.encode(sample)) == sample, "round-trip failed"

## 3. Pack

Documents are concatenated into one token stream separated by EOS, then cut into
fixed-size blocks. No padding, and the result is memory-mapped so the dataset
does not have to fit in RAM.

In [ ]:
BLOCK_SIZE = 512
TOKENS = Path("data/tokens/train.bin")

if not TOKENS.exists():
    pack_corpus(iter_text_file(CORPUS), tokenizer, TOKENS)

dataset = PackedDataset(TOKENS, block_size=BLOCK_SIZE)
train_set, val_set = dataset.split(val_fraction=0.02)

train_loader = DataLoader(train_set, batch_size=8, shuffle=True,
                          collate_fn=collate_packed, drop_last=True)
val_loader = DataLoader(val_set, batch_size=8, collate_fn=collate_packed)

print(f"{dataset.corpus.num_tokens:,} tokens -> {len(dataset):,} blocks")
print(f"train {len(train_set):,} | val {len(val_set):,}")

## 4. Model

`estimate_parameters()` gives the size before anything is allocated, so you can
try shapes cheaply.

In [ ]:
config = AvaConfig.from_preset(
    "hybrid-130m",
    vocab_size=len(tokenizer),
    max_position_embeddings=max(BLOCK_SIZE, 2048),
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
print(f"estimated: {config.estimate_parameters():,} parameters\n")

model = AvaForCausalLM(config)
print(model_summary(model))

## 5. Train

`precision="auto"` picks bf16 wherever it is available. Turn on
`gradient_checkpointing` if you run out of memory — it is the single most
effective knob, at roughly 30% more compute.

In [ ]:
training_config = TrainingConfig(
    max_steps=200,                    # raise substantially for a real run
    learning_rate=3e-4,
    lr_schedule="wsd",
    warmup_ratio=0.02,
    gradient_accumulation_steps=4,    # effective batch = 8 * 4 = 32
    precision="auto",
    gradient_checkpointing=False,
    compile_model=False,              # True is faster after a slow first step
    checkpoint_dir="checkpoints",
    save_every=100,
    eval_every=100,
    log_interval=10,
)

try:
    model, history = train_model(
        model, train_loader, val_loader,
        device=device, training_config=training_config,
    )
except KeyboardInterrupt:
    print("interrupted -- the latest checkpoint is still on disk")

## 6. Curves

In [ ]:
import matplotlib.pyplot as plt

train = [(h["step"], h["train_loss"]) for h in history if "train_loss" in h]
val = [(h["step"], h["val_loss"]) for h in history if "val_loss" in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

ax1.plot(*zip(*train), label="train")
if val:
    ax1.plot(*zip(*val), "o-", label="val")
ax1.set(xlabel="step", ylabel="loss", title="Loss")
ax1.legend()
ax1.grid(alpha=0.3)

throughput = [(h["step"], h["tokens_per_second"]) for h in history
              if h.get("tokens_per_second")]
if throughput:
    ax2.plot(*zip(*throughput), color="tab:green")
ax2.set(xlabel="step", ylabel="tokens/s", title="Throughput")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save and sample

`temperature=0.8, min_p=0.05` is a good default. Min-p sets its threshold
relative to the top token, so it stays tight when the model is confident.

In [ ]:
model.save_pretrained("checkpoints/final")
tokenizer.save_pretrained("checkpoints/final")
print("saved to checkpoints/final")

In [ ]:
model.eval()

generation_config = GenerationConfig(
    max_new_tokens=80,
    temperature=0.8,
    min_p=0.05,
    repetition_penalty=1.05,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

for prompt in ["A state space model", "Attention"]:
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    ids = torch.tensor([[tokenizer.bos_token_id, *ids]], device=device)
    output = model.generate(ids, generation_config=generation_config)
    print("-" * 60)
    print(tokenizer.decode(output[0]))

## Next

- Real data: `scripts/download_corpus.py`, then a much larger `max_steps`.
- Multiple GPUs: `torchrun --nproc_per_node=N scripts/pretrain.py --tokens data/tokens/train.bin`
- Instruction tuning: `ava.data.ChatDataset` with `PaddingCollator` — see [docs/data.md](../docs/data.md).
- Fine-tuning cheaply: `ava.model.lora.apply_lora` — see [docs/generation.md](../docs/generation.md).
